## Task 4: In-Memory HNSW Vector Indexing from Scratch
A simplified Hierarchical Navigable Small World graph: vectors get inserted into a random top layer, greedy nearest-neighbour search descends layer by layer, and the bottom layer holds every point densely connected to its neighbours.


In [2]:
import networkx as nx
from scipy.spatial.distance import cosine
import random
import numpy as np

random.seed(0)
np.random.seed(0)

class SimpleHNSW:
    def __init__(self, dim, M=4, max_level=4, level_mult=0.5):
        self.dim = dim
        self.M = M
        self.max_level = max_level
        self.level_mult = level_mult
        self.layers = [nx.Graph() for _ in range(max_level)]
        self.vectors = {}
        self.entry_point = None

    def _random_level(self):
        lvl = 0
        while random.random() < self.level_mult and lvl < self.max_level - 1:
            lvl += 1
        return lvl

    def _dist(self, a, b):
        return cosine(self.vectors[a], self.vectors[b])

    def insert(self, node_id, vector):
        self.vectors[node_id] = vector
        level = self._random_level()
        for l in range(level + 1):
            self.layers[l].add_node(node_id)

        if self.entry_point is None:
            self.entry_point = node_id
            return

        cur = self.entry_point
        for l in reversed(range(self.max_level)):
            if node_id not in self.layers[l]:
                continue
            candidates = list(self.layers[l].nodes)
            candidates = [c for c in candidates if c != node_id]
            if not candidates:
                continue
            candidates.sort(key=lambda c: self._dist(node_id, c))
            neighbors = candidates[:self.M]
            for nb in neighbors:
                self.layers[l].add_edge(node_id, nb)
            cur = neighbors[0] if neighbors else cur

    def search(self, query_vec, k=3):
        temp_id = "__query__"
        self.vectors[temp_id] = query_vec
        cur = self.entry_point
        for l in reversed(range(self.max_level)):
            if cur not in self.layers[l]:
                continue
            improved = True
            while improved:
                improved = False
                for nb in self.layers[l].neighbors(cur):
                    if self._dist(temp_id, nb) < self._dist(temp_id, cur):
                        cur = nb
                        improved = True
        bottom = list(self.layers[0].nodes)
        bottom = [n for n in bottom if n != temp_id]
        bottom.sort(key=lambda n: self._dist(temp_id, n))
        del self.vectors[temp_id]
        return bottom[:k]

index = SimpleHNSW(dim=8, M=4, max_level=4)
data = {f"v{i}": np.random.randn(8) for i in range(40)}
for k, v in data.items():
    index.insert(k, v)

query = np.random.randn(8)
top_k = index.search(query, k=5)
print("Query nearest neighbours (approximate):", top_k)

# brute-force ground truth for comparison
true_ranked = sorted(data.keys(), key=lambda k: cosine(query, data[k]))[:5]
print("Brute-force nearest neighbours (exact):  ", true_ranked)
print("Overlap with exact search:", len(set(top_k) & set(true_ranked)), "/ 5")

Query nearest neighbours (approximate): ['v31', 'v17', 'v15', 'v30', 'v6']
Brute-force nearest neighbours (exact):   ['v31', 'v17', 'v15', 'v30', 'v6']
Overlap with exact search: 5 / 5


HNSW is *approximate* nearest neighbour search — it trades a small amount of accuracy for much faster lookups than brute-force, which matters once you have millions of vectors. The overlap above shows it's finding most of the true nearest neighbours without comparing against every single point.